# 1 — Math Agent: Reliable Calculation via Tool Use
> An agent that solves multi-step arithmetic problems by calling real functions — never guessing — so every answer is traceable and verifiable.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core langgraph python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
# ── All agent logic ─────────────────────────────────────────────────────────

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


@tool
def add(x: int, y: int) -> int:
    """Add two integers and return their sum."""
    return x + y


@tool
def multiply(x: int, y: int) -> int:
    """Multiply two integers and return their product."""
    return x * y


SYSTEM_PROMPT = SystemMessage(
    "You are a math assistant. Solve problems using only the provided tools. "
    "Do not compute answers yourself."
)


def create_agent():
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return create_react_agent(model=llm, prompt=SYSTEM_PROMPT, tools=[add, multiply])


def run(question: str, agent) -> str:
    """Run a question through the agent and return the final answer."""
    inputs = {"messages": [HumanMessage(question)]}
    final = None
    for step in agent.stream(inputs, stream_mode="values"):
        final = step["messages"][-1]
    return final.content if final else ""


agent = create_agent()
print("Agent ready.")

## The scenario

A sales team of 12 reps closed a strong month. Each rep earns a base of $3,200. Five of them hit their quota and receive a $750 performance bonus on top. We need the total payroll for the month — broken down clearly so finance can verify the numbers before the run.

The agent works through the arithmetic step by step, calling the exact same functions your finance system would use, so every intermediate result is auditable.

In [ ]:
# ── Run the demo ─────────────────────────────────────────────────────────────

QUESTIONS = [
    "What is 8 multiplied by 125?",
    "Add 340 and 876, then multiply the result by 4.",
    (
        "A sales team of 12 reps each earns a base of 3200 per month. "
        "5 of them hit quota and receive a 750 performance bonus. "
        "What is the total monthly payroll?"
    ),
]

print("=" * 60)
print("  MATH AGENT — DEMO RESULTS")
print("=" * 60)

for i, question in enumerate(QUESTIONS, 1):
    answer = run(question, agent)
    print(f"\nScenario {i}")
    print(f"  Input   : {question}")
    print(f"  Answer  : {answer}")
    print("-" * 60)

print("\nDone.")

## Use your own data

Replace the input above with:
- your own arithmetic word problem (multi-step is fine)
- any combination of addition and multiplication expressed in plain English

The agent will return a verified numeric answer, with every intermediate tool call recorded in the trace.